In [0]:

from pyspark.sql import functions as F

# ============================================================
# 1. CONFIGURATION
# ============================================================

silver_table = "retail_ops.silver.orders_clean"

daily_table = "retail_ops.gold.daily_sales"
product_table = "retail_ops.gold.product_performance"
country_table = "retail_ops.gold.country_sales"

# ============================================================
# 2. READ SILVER DATA
# ============================================================

df = spark.table(silver_table)

print("Silver rows:", df.count())

# ============================================================
# 3. DAILY SALES
# ============================================================

daily_sales = (
    df
    .withColumn("sales_date", F.to_date("invoicedate"))
    .groupBy("sales_date")
    .agg(
        F.round(F.sum("revenue"), 2).alias("total_revenue"),
        F.sum("quantity").alias("units_sold"),
        F.countDistinct("invoiceno").alias("total_orders"),
        F.countDistinct("customerid").alias("unique_customers")
    )
    .orderBy("sales_date")
)

(
    daily_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(daily_table)
)

print("Created:", daily_table)

# ============================================================
# 4. PRODUCT PERFORMANCE
# ============================================================

product_performance = (
    df
    .groupBy("stockcode", "description")
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("revenue"), 2).alias("total_revenue"),
        F.countDistinct("invoiceno").alias("total_orders")
    )
    .orderBy(F.desc("total_revenue"))
)

(
    product_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(product_table)
)

print("Created:", product_table)

# ============================================================
# 5. COUNTRY SALES
# ============================================================

country_sales = (
    df
    .groupBy("country")
    .agg(
        F.round(F.sum("revenue"), 2).alias("total_revenue"),
        F.sum("quantity").alias("units_sold"),
        F.countDistinct("invoiceno").alias("total_orders"),
        F.countDistinct("customerid").alias("unique_customers")
    )
    .orderBy(F.desc("total_revenue"))
)

(
    country_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(country_table)
)

print("Created:", country_table)

# ============================================================
# 6. VERIFY GOLD TABLES
# ============================================================

print("Daily sales rows:", spark.table(daily_table).count())
print("Product rows:", spark.table(product_table).count())
print("Country rows:", spark.table(country_table).count())

display(spark.table(daily_table).limit(10))
display(spark.table(product_table).limit(10))
display(spark.table(country_table).limit(10))
